# Fraud Monitoring — Publisher-Level Install Analysis

Hourly install patterns, CPI, and retention (D1/D3) by publisher for a given bundle, OS, and geo.

In [71]:
#@title Colab Authentication

from google.colab import auth
auth.authenticate_user()

In [72]:
#@title Environment Setup
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    try:
        df = client.query(query).result().to_dataframe()
        status = f'✅ {label}: {len(df)} rows' if len(df) > 0 else f'⚠️ {label}: 0 rows'
        print(status)
        return df
    except Exception as e:
        print(f'❌ {label}: {e}')
        return pd.DataFrame()

_pub_name_cache = {}

def enrich_publisher_names(df, bundle_col='publisher_bundle'):
    """Add publisher_name column by looking up dim1_app_with_tracking_publisher_v2."""
    if df.empty or bundle_col not in df.columns:
        return df
    missing = set(df[bundle_col].unique()) - set(_pub_name_cache.keys())
    if missing:
        bids = ", ".join(f"'{b}'" for b in missing)
        q = f"""SELECT DISTINCT app_market_bundle, app_name
                FROM `moloco-ae-view.athena.dim1_app_with_tracking_publisher_v2`
                WHERE app_market_bundle IN ({bids})"""
        try:
            rows = client.query(q).result()
            for r in rows:
                _pub_name_cache[r.app_market_bundle] = r.app_name
        except Exception:
            pass
    df = df.copy()
    df.insert(df.columns.get_loc(bundle_col) + 1, 'publisher_name',
              df[bundle_col].map(_pub_name_cache).fillna(''))
    return df

## Parameters

In [73]:
#@title Parameters
#@markdown **Priority: CAMPAIGN_IDS > MMP_BUNDLE_IDS.**  If campaign IDs are set, bundle/OS/country are derived from them.

CAMPAIGN_IDS_STR = "naoP7qVSZwYbv4GY, unbUzFob1WCmTrVD, jVkXbQ2LIdIKoaQg"  #@param {type:"string"}
MMP_BUNDLE_IDS_STR = ""  #@param {type:"string"}
COUNTRIES_STR = ""  #@param {type:"string"}
LOOKBACK_DAYS = 7  #@param {type:"integer"}

CAMPAIGN_IDS = [s.strip() for s in CAMPAIGN_IDS_STR.split(",") if s.strip()]
MMP_BUNDLE_IDS = [s.strip() for s in MMP_BUNDLE_IDS_STR.split(",") if s.strip()]
COUNTRIES = [s.strip() for s in COUNTRIES_STR.split(",") if s.strip()]

# --- determine mode & build SQL fragments ---
# Output dimension: <OS:Country>  (e.g. "ANDROID:KOR")
# Timezone: KST (UTC+9) for hourly data; daily tables use UTC partition dates.
_MODE = 'campaign' if CAMPAIGN_IDS else 'bundle'
_TZ = 'Asia/Seoul'

_os_country_expr     = "CONCAT(campaign.os, ':', campaign.country)"
_os_country_expr_cv  = "CONCAT(req.device.os, ':', req.device.geo.country)"

if _MODE == 'campaign':
    _ids = ", ".join(f"'{c}'" for c in CAMPAIGN_IDS)
    _where_filter = f"AND campaign_id IN ({_ids})"
    _where_filter_cv = f"AND api.campaign.id IN ({_ids})"
    _extra_select = f"campaign_id, {_os_country_expr} AS os_country,"
    _extra_select_cv = f"api.campaign.id AS campaign_id, {_os_country_expr_cv} AS os_country,"
    _extra_group = f"campaign_id, {_os_country_expr},"
    _extra_group_cv = f"api.campaign.id, {_os_country_expr_cv},"
    _dim_cols = ['campaign_id', 'os_country']
else:
    _bids = ", ".join(f"'{b}'" for b in MMP_BUNDLE_IDS)
    _where_filter = f"AND advertiser.mmp_bundle_id IN ({_bids})"
    _where_filter_cv = f"AND api.product.app.tracking_bundle IN ({_bids})"
    _extra_select = f"{_os_country_expr} AS os_country,"
    _extra_select_cv = f"{_os_country_expr_cv} AS os_country,"
    _extra_group = f"{_os_country_expr},"
    _extra_group_cv = f"{_os_country_expr_cv},"
    _dim_cols = ['os_country']

if COUNTRIES:
    _clist = ", ".join(f"'{c}'" for c in COUNTRIES)
    _where_filter += f"\n  AND campaign.country IN ({_clist})"
    _where_filter_cv += f"\n  AND req.device.geo.country IN ({_clist})"

print(f'Mode:      {_MODE}')
if _MODE == 'campaign':
    print(f'Campaigns: {CAMPAIGN_IDS}')
else:
    print(f'Bundles:   {MMP_BUNDLE_IDS}')
print(f'Countries: {COUNTRIES or "(all)"}')
print(f'Timezone:  KST (UTC+9)')
print(f'Lookback:  {LOOKBACK_DAYS} days')

Mode:      campaign
Campaigns: ['naoP7qVSZwYbv4GY', 'unbUzFob1WCmTrVD', 'jVkXbQ2LIdIKoaQg']
Countries: (all)
Timezone:  KST (UTC+9)
Lookback:  7 days


---

In [74]:
# placeholder

---
## 1. Daily Publisher Summary (installs, CPI, retention)

In [75]:
#1. Daily Publisher Summary — installs, CPI, D1/D3 retention

q_pub = f"""
SELECT
  {_extra_select}
  date_utc,
  publisher.app_market_bundle AS publisher_bundle,
  SUM(installs) AS installs,
  SUM(installs_rejected) AS installs_rejected,
  ROUND(SUM(gross_spend_usd), 2) AS spend_usd,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), NULLIF(SUM(installs), 0)), 2) AS cpi,
  SUM(impressions) AS impressions,
  SUM(clicks) AS clicks,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d1), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d1_pct,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d3), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d3_pct,
  ROUND(SAFE_DIVIDE(SUM(clicks), NULLIF(SUM(impressions), 0)) * 100, 2) AS ctr_pct
FROM `moloco-ae-view.athena.fact_dsp_publisher`
WHERE date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY) AND CURRENT_DATE()
  {_where_filter}
GROUP BY {_extra_group} date_utc, publisher.app_market_bundle
HAVING installs > 0
ORDER BY date_utc DESC, installs DESC
"""

df_pub = run_query(q_pub, '1. Daily Publisher Summary')
if not df_pub.empty:
    df_pub = enrich_publisher_names(df_pub)
    numeric_cols = ['installs', 'installs_rejected', 'spend_usd', 'cpi', 'impressions', 'clicks',
                    'retention_d1_pct', 'retention_d3_pct', 'ctr_pct']
    for col in numeric_cols:
        if col in df_pub.columns:
            df_pub[col] = pd.to_numeric(df_pub[col], errors='coerce')

    for oc in sorted(df_pub['os_country'].unique()):
        sub = df_pub[df_pub['os_country'] == oc]
        agg = sub.groupby(['publisher_bundle', 'publisher_name']).agg(
            total_installs=('installs', 'sum'),
            total_spend=('spend_usd', 'sum'),
            total_impressions=('impressions', 'sum'),
            total_clicks=('clicks', 'sum'),
        ).sort_values('total_installs', ascending=False)
        agg['cpi'] = (agg['total_spend'] / agg['total_installs']).round(2)
        agg['ctr_pct'] = (agg['total_clicks'] / agg['total_impressions'] * 100).round(2)

        print(f'\n  [{oc}] Top publishers (L{LOOKBACK_DAYS}D):')
        for (pub, name), row in agg.head(10).iterrows():
            label = f'{name[:30]} ({pub[:30]})' if name else pub[:50]
            print(f'    {label:65s}  installs={row["total_installs"]:>6,.0f}  CPI=${row["cpi"]:.2f}  CTR={row["ctr_pct"]:.2f}%')
df_pub

✅ 1. Daily Publisher Summary: 1064 rows

  [ANDROID:ARE] Top publishers (L7D):
    com.auto.war.crash.arena                            installs=     1  CPI=$0.00  CTR=100.00%
    com.rheia.patrino                                   installs=     1  CPI=$0.11  CTR=50.00%
    qiblafinder.prayertimes.qibladirection.hijricalend  installs=     1  CPI=$0.04  CTR=100.00%
    me.pokeraid                                         installs=     1  CPI=$0.00  CTR=0.00%
    info.t4w.vp                                         installs=     1  CPI=$0.21  CTR=12.50%
    com.wool.puzzle.game3d                              installs=     1  CPI=$0.22  CTR=20.00%
    com.studion.mergearena                              installs=     1  CPI=$0.02  CTR=42.86%
    com.scores365                                       installs=     1  CPI=$0.72  CTR=11.11%
    com.multicastgames.butcher                          installs=     1  CPI=$2.73  CTR=25.58%
    com.battle.worldofartillery                         installs=

,campaign_id,os_country,date_utc,publisher_bundle,installs,installs_rejected,spend_usd,cpi,impressions,clicks,retention_d1_pct,retention_d3_pct,ctr_pct
0,jVkXbQ2LIdIKoaQg,ANDROID:IDN,2026-03-04,com.traverse.zhfx.en.gp,4,0,12.55,3.14,284,149,0.0,0.0,52.46
1,jVkXbQ2LIdIKoaQg,ANDROID:IDN,2026-03-04,com.worldance.drama,2,0,0.96,0.48,89,26,0.0,0.0,29.21
2,naoP7qVSZwYbv4GY,ANDROID:BEL,2026-03-04,com.ttee.leeplayer,1,0,0.00,0.00,0,0,0.0,0.0,NaN
3,unbUzFob1WCmTrVD,ANDROID:FRA,2026-03-04,com.bag.fight.backpack.critter.battle.rumble.paws,1,0,0.00,0.00,0,0,0.0,0.0,NaN
4,jVkXbQ2LIdIKoaQg,ANDROID:IDN,2026-03-04,com.livescore,1,0,0.00,0.00,7,0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1059,naoP7qVSZwYbv4GY,ANDROID:SWE,2026-03-03,defense.roguelike.cell.shoot.survivor,1,0,1.15,1.15,72,24,0.0,0.0,33.33
1060,naoP7qVSZwYbv4GY,ANDROID:FRA,2026-03-03,org.smapps.digger,1,0,0.88,0.88,106,28,0.0,0.0,26.42
1061,naoP7qVSZwYbv4GY,ANDROID:SAU,2026-03-03,com.king.knightsrage,1,0,0.08,0.08,6,2,0.0,0.0,33.33
1062,unbUzFob1WCmTrVD,ANDROID:DEU,2026-03-03,wool.match.color.sort.jam.puzzle,1,0,0.77,0.77,37,4,0.0,0.0,10.81


## 2. Hourly Install Patterns by Publisher

In [76]:
#2. Hourly Install Trend by Publisher (last 24h, from cv postback)

q_hourly = f"""
SELECT
  {_extra_select_cv}
  DATETIME_TRUNC(DATETIME(timestamp, '{_TZ}'), HOUR) AS hour_kst,
  req.app.bundle AS publisher_bundle,
  COUNT(*) AS installs
FROM `focal-elf-631.prod_stream_view.cv`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND UPPER(cv.event) = 'INSTALL'
  {_where_filter_cv}
GROUP BY {_extra_group_cv} hour_kst, publisher_bundle
ORDER BY hour_kst ASC, installs DESC
"""

df_hourly = run_query(q_hourly, '2. Hourly Install Trend (24h)')

TOP_N_COUNTRIES = 5

if not df_hourly.empty:
    df_hourly = enrich_publisher_names(df_hourly)
    df_hourly['pub_label'] = df_hourly.apply(
        lambda r: f'{r["publisher_name"][:25]} ({r["publisher_bundle"][:25]})' if r['publisher_name'] else r['publisher_bundle'], axis=1)
    df_hourly['installs'] = pd.to_numeric(df_hourly['installs'], errors='coerce')
    df_hourly['hour_kst'] = pd.to_datetime(df_hourly['hour_kst'])

    import math
    HOUR_MS = 3_600_000 * 0.8

    top_oc = (df_hourly.groupby('os_country')['installs'].sum()
              .sort_values(ascending=False).head(TOP_N_COUNTRIES).index.tolist())
    all_oc = sorted(df_hourly['os_country'].unique())
    if len(all_oc) > TOP_N_COUNTRIES:
        print(f'  {len(all_oc)} OS:Country segments found — showing top {TOP_N_COUNTRIES} by install volume: {top_oc}')
        print(f'  (skipped: {[c for c in all_oc if c not in top_oc]})')

    for oc in top_oc:
        oc_df = df_hourly[df_hourly['os_country'] == oc]
        top_pubs = oc_df.groupby('publisher_bundle')['installs'].sum().sort_values(ascending=False).head(10).index.tolist()
        chart_df = oc_df[oc_df['publisher_bundle'].isin(top_pubs)].copy()
        chart_df['hour_kst'] = chart_df['hour_kst'].astype(str)

        fig = px.bar(chart_df, x='hour_kst', y='installs', color='pub_label',
                     title=f'[{oc}] Hourly Install Trend (Top 10, last 24h, KST)',
                     labels={'hour_kst': 'Time (KST)', 'installs': 'Installs', 'pub_label': 'Publisher'})
        fig.update_traces(width=HOUR_MS)
        fig.update_layout(barmode='stack', height=500, xaxis=dict(tickformat='%b %d %H:%M', dtick=3_600_000))
        fig.show()

        hourly_total = chart_df.groupby('hour_kst')['installs'].transform('sum')
        chart_df['pct'] = (chart_df['installs'] / hourly_total * 100).round(1)
        fig_pct = px.bar(chart_df, x='hour_kst', y='pct', color='pub_label',
                         title=f'[{oc}] Hourly Install Share (Top 10, last 24h, KST)',
                         labels={'hour_kst': 'Time (KST)', 'pct': '% of Installs', 'pub_label': 'Publisher'})
        fig_pct.update_traces(width=HOUR_MS)
        fig_pct.update_layout(barmode='stack', height=500,
                              xaxis=dict(tickformat='%b %d %H:%M', dtick=3_600_000),
                              yaxis=dict(range=[0, 100], dtick=20, ticksuffix='%'))
        fig_pct.show()

        for pub in top_pubs[:3]:
            pub_df = chart_df[chart_df['publisher_bundle'] == pub]
            if pub_df.empty:
                continue
            pub_label = pub_df['pub_label'].iloc[0]
            fig2 = px.bar(pub_df, x='hour_kst', y='installs',
                          title=f'[{oc}] {pub_label[:60]}',
                          labels={'hour_kst': 'Time (KST)', 'installs': 'Installs'})
            fig2.update_traces(width=HOUR_MS)
            max_val = pub_df['installs'].max()
            y_dtick = max(1, math.ceil(max_val / 6)) if max_val > 0 else 1
            fig2.update_layout(height=300, xaxis=dict(tickformat='%b %d %H:%M', dtick=3_600_000),
                               yaxis=dict(dtick=y_dtick, tick0=0))
            fig2.show()
df_hourly

✅ 2. Hourly Install Trend (24h): 1210 rows
  26 OS:Country segments found — showing top 5 by install volume: ['ANDROID:FRA', 'ANDROID:DEU', 'ANDROID:IDN', 'ANDROID:POL', 'ANDROID:GBR']
  (skipped: ['ANDROID:ARE', 'ANDROID:AUS', 'ANDROID:AUT', 'ANDROID:BEL', 'ANDROID:CAN', 'ANDROID:CHE', 'ANDROID:CYP', 'ANDROID:ESP', 'ANDROID:HKG', 'ANDROID:HUN', 'ANDROID:IRL', 'ANDROID:ITA', 'ANDROID:KWT', 'ANDROID:MYS', 'ANDROID:NLD', 'ANDROID:NOR', 'ANDROID:NZL', 'ANDROID:QAT', 'ANDROID:SAU', 'ANDROID:SGP', 'ANDROID:SWE'])


,campaign_id,os_country,hour_kst,publisher_bundle,installs
0,naoP7qVSZwYbv4GY,ANDROID:CAN,2026-03-03 14:00:00,com.pc.sand.loop,1
1,naoP7qVSZwYbv4GY,ANDROID:SGP,2026-03-03 14:00:00,com.and.imi.hcw,1
2,naoP7qVSZwYbv4GY,ANDROID:DEU,2026-03-03 14:00:00,com.wasteland.heart,1
3,naoP7qVSZwYbv4GY,ANDROID:FRA,2026-03-03 14:00:00,com.jedi.renovation,1
4,naoP7qVSZwYbv4GY,ANDROID:FRA,2026-03-03 14:00:00,com.xq.archeroii,1
...,...,...,...,...,...
1205,naoP7qVSZwYbv4GY,ANDROID:POL,2026-03-04 13:00:00,com.innplaylabs.animalkingdomraid,1
1206,naoP7qVSZwYbv4GY,ANDROID:DEU,2026-03-04 13:00:00,com.TreetopCrew.EarthInc,1
1207,unbUzFob1WCmTrVD,ANDROID:DEU,2026-03-04 13:00:00,com.onicore.backpack.rush,1
1208,naoP7qVSZwYbv4GY,ANDROID:DEU,2026-03-04 13:00:00,com.weaver.app.prod,1


## 3. D1 / D3 Retention by Publisher

In [77]:
# 3a. OS:Country-level Retention Summary (no publisher breakdown)

q_ret_summary = f"""
SELECT
  {_extra_select}
  SUM(installs) AS installs,
  ROUND(SUM(gross_spend_usd), 2) AS spend_usd,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), NULLIF(SUM(installs), 0)), 2) AS cpi,
  ROUND(SUM(retained_users_d1), 1) AS retained_d1,
  ROUND(SUM(retained_users_d3), 1) AS retained_d3,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d1), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d1_pct,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d3), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d3_pct
FROM `moloco-ae-view.athena.fact_dsp_publisher`
WHERE date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY) AND DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  {_where_filter}
GROUP BY {_extra_group[:-1]}
HAVING installs > 0
ORDER BY installs DESC
"""

df_ret_summary = run_query(q_ret_summary, '3a. Retention by OS:Country')
if not df_ret_summary.empty:
    for col in df_ret_summary.columns:
        if col not in _dim_cols:
            df_ret_summary[col] = pd.to_numeric(df_ret_summary[col], errors='coerce')
    print(f'\n  Retention summary by OS:Country (L{LOOKBACK_DAYS}D, excl. today):')
    display(df_ret_summary)

# 3b. D1/D3 Retention by Publisher (per OS:Country)

q_ret = f"""
SELECT
  {_extra_select}
  publisher.app_market_bundle AS publisher_bundle,
  SUM(installs) AS installs,
  ROUND(SUM(gross_spend_usd), 2) AS spend_usd,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), NULLIF(SUM(installs), 0)), 2) AS cpi,
  ROUND(SUM(retained_users_d1), 1) AS retained_d1,
  ROUND(SUM(retained_users_d3), 1) AS retained_d3,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d1), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d1_pct,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d3), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d3_pct
FROM `moloco-ae-view.athena.fact_dsp_publisher`
WHERE date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY) AND DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  {_where_filter}
GROUP BY {_extra_group} publisher.app_market_bundle
HAVING installs >= 1
ORDER BY installs DESC
"""

df_ret = run_query(q_ret, '3b. D1/D3 Retention by Publisher')
if not df_ret.empty:
    df_ret = enrich_publisher_names(df_ret)
    str_cols = _dim_cols + ['publisher_bundle', 'publisher_name']
    for col in df_ret.columns:
        if col not in str_cols:
            df_ret[col] = pd.to_numeric(df_ret[col], errors='coerce')

    for oc in sorted(df_ret['os_country'].unique()):
        sub = df_ret[df_ret['os_country'] == oc].sort_values('installs', ascending=False)
        overall_d1 = (sub['retained_d1'].sum() / sub['installs'].sum() * 100) if sub['installs'].sum() > 0 else 0
        overall_d3 = (sub['retained_d3'].sum() / sub['installs'].sum() * 100) if sub['installs'].sum() > 0 else 0
        p10_d1 = sub['retention_d1_pct'].quantile(0.10)
        p10_cpi = sub[sub['cpi'] > 0]['cpi'].quantile(0.10) if (sub['cpi'] > 0).any() else 0
        print(f'\n  [{oc}] {len(sub)} publishers  |  D1={overall_d1:.1f}%  D3={overall_d3:.1f}%  |  P10 D1={p10_d1:.1f}%  P10 CPI=${p10_cpi:.2f}')
        display(sub.head(20).reset_index(drop=True))

    fig = px.scatter(df_ret[df_ret['installs'] >= 3], x='cpi', y='retention_d1_pct', size='installs',
                     hover_name='publisher_bundle', color='os_country',
                     title=f'Publisher CPI vs D1 Retention by OS:Country (L{LOOKBACK_DAYS}D, installs>=3)',
                     labels={'cpi': 'CPI ($)', 'retention_d1_pct': 'D1 Retention %', 'installs': 'Installs'},
                     hover_data={'retention_d3_pct': ':.1f', 'spend_usd': ':,.2f'})
    fig.update_layout(height=500)
    fig.show()

✅ 3a. Retention by OS:Country: 46 rows

  Retention summary by OS:Country (L7D, excl. today):


,campaign_id,os_country,installs,spend_usd,cpi,retained_d1,retained_d3,retention_d1_pct,retention_d3_pct
0,naoP7qVSZwYbv4GY,ANDROID:FRA,173,241.64,1.40,0.0,0.0,0.0,0.0
1,naoP7qVSZwYbv4GY,ANDROID:DEU,123,215.52,1.75,0.0,0.0,0.0,0.0
2,jVkXbQ2LIdIKoaQg,ANDROID:IDN,123,804.30,6.54,0.0,0.0,0.0,0.0
3,naoP7qVSZwYbv4GY,ANDROID:POL,109,166.43,1.53,0.0,0.0,0.0,0.0
4,naoP7qVSZwYbv4GY,ANDROID:GBR,65,82.94,1.28,0.0,0.0,0.0,0.0
5,unbUzFob1WCmTrVD,ANDROID:DEU,49,535.21,10.92,0.0,0.0,0.0,0.0
6,naoP7qVSZwYbv4GY,ANDROID:HUN,44,51.67,1.17,0.0,0.0,0.0,0.0
7,jVkXbQ2LIdIKoaQg,ANDROID:MYS,34,556.87,16.38,0.0,0.0,0.0,0.0
8,jVkXbQ2LIdIKoaQg,ANDROID:ITA,31,236.31,7.62,0.0,0.0,0.0,0.0
9,unbUzFob1WCmTrVD,ANDROID:FRA,31,283.43,9.14,0.0,0.0,0.0,0.0


✅ 3b. D1/D3 Retention by Publisher: 953 rows

  [ANDROID:ARE] 17 publishers  |  D1=0.0%  D3=0.0%  |  P10 D1=0.0%  P10 CPI=$0.03


,campaign_id,os_country,publisher_bundle,installs,spend_usd,cpi,retained_d1,retained_d3,retention_d1_pct,retention_d3_pct
0,naoP7qVSZwYbv4GY,ANDROID:ARE,ragdoll.hit.stickman,1,0.04,0.04,0.0,0.0,0.0,0.0
1,unbUzFob1WCmTrVD,ANDROID:ARE,me.pokeraid,1,0.00,0.00,0.0,0.0,0.0,0.0
2,unbUzFob1WCmTrVD,ANDROID:ARE,com.multicastgames.butcher,1,2.73,2.73,0.0,0.0,0.0,0.0
3,naoP7qVSZwYbv4GY,ANDROID:ARE,com.middleeast.huntingsniper,1,0.05,0.05,0.0,0.0,0.0,0.0
4,naoP7qVSZwYbv4GY,ANDROID:ARE,com.wool.puzzle.game3d,1,0.22,0.22,0.0,0.0,0.0,0.0
5,naoP7qVSZwYbv4GY,ANDROID:ARE,com.loomgames.pixelflow,1,0.25,0.25,0.0,0.0,0.0,0.0
6,naoP7qVSZwYbv4GY,ANDROID:ARE,com.battle.worldofartillery,1,0.08,0.08,0.0,0.0,0.0,0.0
7,naoP7qVSZwYbv4GY,ANDROID:ARE,com.auto.war.crash.arena,1,0.00,0.00,0.0,0.0,0.0,0.0
8,naoP7qVSZwYbv4GY,ANDROID:ARE,com.hitv.savita,1,0.06,0.06,0.0,0.0,0.0,0.0
9,naoP7qVSZwYbv4GY,ANDROID:ARE,qiblafinder.prayertimes.qibladirection.hijricalendar,1,0.04,0.04,0.0,0.0,0.0,0.0



  [ANDROID:AUS] 12 publishers  |  D1=0.0%  D3=0.0%  |  P10 D1=0.0%  P10 CPI=$0.06


...

,campaign_id,os_country,publisher_bundle,installs,spend_usd,cpi,retained_d1,retained_d3,retention_d1_pct,retention_d3_pct
0,naoP7qVSZwYbv4GY,ANDROID:SAU,io.supercent.weaponrpg,2,0.94,0.47,0.0,0.0,0.0,0.0
1,naoP7qVSZwYbv4GY,ANDROID:SAU,com.qxgame.ghard,1,0.05,0.05,0.0,0.0,0.0,0.0
2,unbUzFob1WCmTrVD,ANDROID:SAU,com.aidis.lastcloudiaen,1,0.05,0.05,0.0,0.0,0.0,0.0
3,naoP7qVSZwYbv4GY,ANDROID:SAU,com.habby.archero,1,0.05,0.05,0.0,0.0,0.0,0.0
4,naoP7qVSZwYbv4GY,ANDROID:SAU,shelter.hero.hapyx,1,0.30,0.30,0.0,0.0,0.0,0.0
5,naoP7qVSZwYbv4GY,ANDROID:SAU,com.aftaa.magicarcher,1,0.33,0.33,0.0,0.0,0.0,0.0
6,naoP7qVSZwYbv4GY,ANDROID:SAU,com.zzsjus.google,1,0.16,0.16,0.0,0.0,0.0,0.0
7,naoP7qVSZwYbv4GY,ANDROID:SAU,train.ride.master,1,0.05,0.05,0.0,0.0,0.0,0.0
8,naoP7qVSZwYbv4GY,ANDROID:SAU,wp.wattpad,1,0.03,0.03,0.0,0.0,0.0,0.0
9,naoP7qVSZwYbv4GY,ANDROID:SAU,com.LuB.DeliveryConstruct,1,0.04,0.04,0.0,0.0,0.0,0.0



  [ANDROID:SGP] 20 publishers  |  D1=0.0%  D3=0.0%  |  P10 D1=0.0%  P10 CPI=$0.02


,campaign_id,os_country,publisher_bundle,installs,spend_usd,cpi,retained_d1,retained_d3,retention_d1_pct,retention_d3_pct
0,naoP7qVSZwYbv4GY,ANDROID:SGP,mech.fortress.tower.defense.hero.war,2,0.25,0.12,0.0,0.0,0.0,0.0
1,unbUzFob1WCmTrVD,ANDROID:SGP,com.tp.pmu3d,1,0.87,0.87,0.0,0.0,0.0,0.0
2,naoP7qVSZwYbv4GY,ANDROID:SGP,com.nexon.ma,1,0.14,0.14,0.0,0.0,0.0,0.0
3,unbUzFob1WCmTrVD,ANDROID:SGP,tv.twitch.android.app,1,0.01,0.01,0.0,0.0,0.0,0.0
4,naoP7qVSZwYbv4GY,ANDROID:SGP,com.thecarousell.Carousell,1,0.00,0.00,0.0,0.0,0.0,0.0
5,naoP7qVSZwYbv4GY,ANDROID:SGP,kr.co.vcnc.android.couple,1,0.00,0.00,0.0,0.0,0.0,0.0
6,unbUzFob1WCmTrVD,ANDROID:SGP,com.michatapp.im,1,0.95,0.95,0.0,0.0,0.0,0.0
7,naoP7qVSZwYbv4GY,ANDROID:SGP,com.vitastudio.mahjong,1,0.94,0.94,0.0,0.0,0.0,0.0
8,naoP7qVSZwYbv4GY,ANDROID:SGP,com.chamelio.alcatmist,1,0.01,0.01,0.0,0.0,0.0,0.0
9,unbUzFob1WCmTrVD,ANDROID:SGP,com.master.runner,1,0.36,0.36,0.0,0.0,0.0,0.0



  [ANDROID:SWE] 18 publishers  |  D1=0.0%  D3=0.0%  |  P10 D1=0.0%  P10 CPI=$0.02


,campaign_id,os_country,publisher_bundle,installs,spend_usd,cpi,retained_d1,retained_d3,retention_d1_pct,retention_d3_pct
0,unbUzFob1WCmTrVD,ANDROID:SWE,com.pixelfederation.ts3.train.station.evolution.building.manager.tycoon.simu...,1,0.46,0.46,0.0,0.0,0.0,0.0
1,unbUzFob1WCmTrVD,ANDROID:SWE,games.resourcer.hex,1,0.49,0.49,0.0,0.0,0.0,0.0
2,naoP7qVSZwYbv4GY,ANDROID:SWE,defense.roguelike.cell.shoot.survivor,1,1.15,1.15,0.0,0.0,0.0,0.0
3,unbUzFob1WCmTrVD,ANDROID:SWE,com.king.knightsrage,1,2.21,2.21,0.0,0.0,0.0,0.0
4,unbUzFob1WCmTrVD,ANDROID:SWE,com.multicastgames.butcher,1,1.87,1.87,0.0,0.0,0.0,0.0
5,naoP7qVSZwYbv4GY,ANDROID:SWE,com.sled.surfers.game,1,0.21,0.21,0.0,0.0,0.0,0.0
6,naoP7qVSZwYbv4GY,ANDROID:SWE,com.diamondlife.slots.vegas.free,1,0.17,0.17,0.0,0.0,0.0,0.0
7,unbUzFob1WCmTrVD,ANDROID:SWE,com.Overcurve.Corebound,1,0.09,0.09,0.0,0.0,0.0,0.0
8,naoP7qVSZwYbv4GY,ANDROID:SWE,com.Beauchamp.Messenger.external,1,0.09,0.09,0.0,0.0,0.0,0.0
9,naoP7qVSZwYbv4GY,ANDROID:SWE,com.bighelmets.destiny,1,0.06,0.06,0.0,0.0,0.0,0.0


---
## 4. Suspicious Publisher Flags

In [78]:
# 4. Suspicious Publisher Flags (per OS:Country, percentile-based)

MIN_INSTALLS = 10
flags = []

if not df_ret.empty:
    for oc in sorted(df_ret['os_country'].unique()):
        sub = df_ret[df_ret['os_country'] == oc]
        p10_d1 = sub['retention_d1_pct'].quantile(0.10)
        p10_cpi = sub['cpi'].quantile(0.10)

        for _, row in sub.iterrows():
            pub = row['publisher_bundle']
            pub_name = row.get('publisher_name', '')
            reasons = []
            if row['installs'] >= MIN_INSTALLS and row['retention_d1_pct'] < p10_d1:
                reasons.append(f'D1 {row["retention_d1_pct"]:.1f}% < P10 threshold {p10_d1:.1f}%')
            if row['retention_d1_pct'] == 0 and row['installs'] >= 20:
                reasons.append(f'Zero D1 retention with {row["installs"]:.0f} installs')
            if row['installs'] >= MIN_INSTALLS and row['cpi'] < p10_cpi:
                reasons.append(f'CPI ${row["cpi"]:.2f} < P10 threshold ${p10_cpi:.2f}')
            if reasons:
                flags.append({'os_country': oc, 'publisher': pub, 'publisher_name': pub_name,
                              'installs': row['installs'], 'cpi': row['cpi'],
                              'd1_ret': row['retention_d1_pct'], 'reasons': reasons})

if not df_hourly.empty:
    pub_hour_agg = df_hourly.groupby(['os_country', 'publisher_bundle']).agg(
        total=('installs', 'sum'), hours_active=('hour_kst', 'nunique')).reset_index()
    for _, row in pub_hour_agg.iterrows():
        if row['total'] >= 20 and row['hours_active'] <= 2:
            existing = next((f for f in flags if f['publisher'] == row['publisher_bundle']
                             and f.get('os_country') == row['os_country']), None)
            reason = f'Installs concentrated in {row["hours_active"]} hour(s) only'
            if existing:
                existing['reasons'].append(reason)
            else:
                flags.append({'os_country': row['os_country'], 'publisher': row['publisher_bundle'],
                              'installs': row['total'], 'cpi': None, 'd1_ret': None, 'reasons': [reason]})

mode_label = f'Campaigns: {CAMPAIGN_IDS}' if _MODE == 'campaign' else f'Bundles: {MMP_BUNDLE_IDS}'
print(f'{"=" * 60}')
print(f'  Fraud Monitoring Summary')
print(f'  {mode_label} / L{LOOKBACK_DAYS}D')
print(f'{"=" * 60}')

print('\n  Flagging Criteria (per OS:Country, P10 percentile):')
print(f'    1. D1 retention below P10 of segment  (min {MIN_INSTALLS} installs)')
print(f'    2. Zero D1 retention                   (min 20 installs)')
print(f'    3. CPI below P10 of segment            (min {MIN_INSTALLS} installs)')
print(f'    4. Installs in <=2 hours over 24h      (min 20 installs)')

if not df_ret.empty:
    print('\n  Percentile thresholds used:')
    for oc in sorted(df_ret['os_country'].unique()):
        sub = df_ret[df_ret['os_country'] == oc]
        print(f'    [{oc}] D1 P10={sub["retention_d1_pct"].quantile(0.10):.1f}%  CPI P10=${sub["cpi"].quantile(0.10):.2f}')

if flags:
    print(f'\n  {len(flags)} publisher(s) flagged:\n')
    for oc in sorted(set(f['os_country'] for f in flags)):
        oc_flags = [f for f in flags if f['os_country'] == oc]
        print(f'  --- [{oc}] ({len(oc_flags)} flagged) ---')
        for f in sorted(oc_flags, key=lambda x: x['installs'], reverse=True):
            cpi_str = f'${f["cpi"]:.2f}' if f['cpi'] is not None else 'N/A'
            d1_str = f'{f["d1_ret"]:.1f}%' if f['d1_ret'] is not None else 'N/A'
            name = f.get('publisher_name', '')
            label = f'{name} ({f["publisher"]})' if name else f['publisher']
            print(f'  {label[:65]}')
            print(f'    Installs={f["installs"]:.0f}  CPI={cpi_str}  D1={d1_str}')
            for r in f['reasons']:
                print(f'    >> {r}')
        print()
else:
    print('\n  No suspicious publishers flagged')

  Fraud Monitoring Summary
  Campaigns: ['naoP7qVSZwYbv4GY', 'unbUzFob1WCmTrVD', 'jVkXbQ2LIdIKoaQg'] / L7D

  Flagging Criteria (per OS:Country, P10 percentile):
    1. D1 retention below P10 of segment  (min 10 installs)
    2. Zero D1 retention                   (min 20 installs)
    3. CPI below P10 of segment            (min 10 installs)
    4. Installs in <=2 hours over 24h      (min 20 installs)

  Percentile thresholds used:
    [ANDROID:ARE] D1 P10=0.0%  CPI P10=$0.01
    [ANDROID:AUS] D1 P10=0.0%  CPI P10=$0.04
    [ANDROID:AUT] D1 P10=0.0%  CPI P10=$0.04
    [ANDROID:BEL] D1 P10=0.0%  CPI P10=$0.04
    [ANDROID:CAN] D1 P10=0.0%  CPI P10=$0.06
    [ANDROID:CHE] D1 P10=0.0%  CPI P10=$0.03
    [ANDROID:CYP] D1 P10=0.0%  CPI P10=$0.02
    [ANDROID:DEU] D1 P10=0.0%  CPI P10=$0.05
    [ANDROID:ESP] D1 P10=0.0%  CPI P10=$0.00
    [ANDROID:FRA] D1 P10=0.0%  CPI P10=$0.05
    [ANDROID:GBR] D1 P10=0.0%  CPI P10=$0.03
    [ANDROID:HKG] D1 P10=0.0%  CPI P10=$0.24
    [ANDROID:HUN] D1 P10